# Lesson 7: Evaluating Agentic RAG Systems

前面的课程里已经有普通 RAG 的评测思路，但 `Agentic RAG` 还多了几层需要评估的对象：

- 计划是否合理
- 工具选得对不对
- 证据是否扎实
- 是否为了一个简单问题做了过多无效步骤

这一节补的就是：

- `agentic RAG evaluation framework`

也就是：不仅评答案，还评整个检索与工具使用过程。


## 这一节的技术在做什么

这一节讲的是：

- `agentic RAG evaluation framework`

也就是：当系统已经不是普通 RAG，而是一个会规划、会调工具、会多步检索的 agent 时，我们该怎么评测它。

在普通 RAG 里，评测往往聚焦于：

- 最终答案是否正确
- 是否 grounded
- 检索结果是否相关

但到了 Agentic RAG，这些还不够，因为系统已经多出了新的可变因素：

- 计划是否合理
- 工具选得是否合适
- 是否做了多余的检索
- 是否把简单问题搞得过于复杂
- 证据采集过程是否高效

所以 Agentic RAG 的评测对象，不再只是“最后一句答案”，而是：

1. 规划质量
2. 工具调用质量
3. 检索证据质量
4. 最终答案质量
5. 整体效率

这就是为什么这一节很重要。

因为如果没有这层评测，你只能得到一种很模糊的反馈：

- “答案好像还行”

但你并不知道：

- 它到底是规划得好
- 还是只是运气好
- 是证据找得准
- 还是最终模型很会润色
- 是系统真的高效
- 还是只是做了很多无效步骤

这节 notebook 的目标，就是把这些维度拆开看。

你可以把它理解成：

- `Lesson 5` 讲“怎么规划”
- `Lesson 6` 讲“怎么反思和重试”
- `Lesson 7` 讲“怎么评估整个 Agentic RAG 系统是否真的做得好”

这一节的核心收获是：

- Agentic RAG 的评测不是只看答案
- 而是要评估“规划 -> 检索 -> 工具调用 -> 综合回答”这一整条链


## Setup


In [ ]:
import json
import os
import re
import sys
from pathlib import Path
from typing import Any, Optional

import nest_asyncio
from pydantic import BaseModel, Field

nest_asyncio.apply()


In [ ]:
# 这一组 notebook 不是重新造一套数据，而是直接复用 Lesson 4 里的论文和工具工厂。
# 这样你学到的是“在原有 Agentic RAG 系统上继续加高级模式”。

lesson4_dir = (Path.cwd() / "../Lesson_4").resolve()
if str(lesson4_dir) not in sys.path:
    sys.path.append(str(lesson4_dir))

from helper import get_dashscope_api_key
from utils import get_doc_tools

from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.openai_like import OpenAILike


def find_workspace_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if candidate.name == "AIAgent":
            return candidate
    raise FileNotFoundError("Could not find the AIAgent workspace root.")


def find_local_bge_snapshot() -> Path:
    workspace_root = find_workspace_root()
    snapshot_root = workspace_root / "models" / "models--BAAI--bge-small-en-v1.5" / "snapshots"
    snapshots = sorted(
        path for path in snapshot_root.iterdir()
        if path.is_dir() and (path / "config.json").exists()
    )
    if not snapshots:
        raise FileNotFoundError(f"No valid local BGE snapshot found under {snapshot_root}")
    return snapshots[0]


# 继续沿用课程中的 qwen-max + 本地 BGE embedding 配置。
llm = OpenAILike(
    api_key=get_dashscope_api_key(),
    api_base="https://dashscope.aliyuncs.com/compatible-mode/v1",
    model="qwen-max",
    temperature=0.1,
    context_window=128000,
    is_chat_model=True,
    is_function_calling_model=True,
)
Settings.llm = llm
Settings.embed_model = HuggingFaceEmbedding(
    model_name=str(find_local_bge_snapshot()),
    device="cpu",
)


papers = [
    "metagpt.pdf",
    "longlora.pdf",
    "selfrag.pdf",
]


def build_tool_registry() -> dict[str, Any]:
    registry = {}
    for paper in papers:
        paper_name = Path(paper).stem
        paper_path = lesson4_dir / paper
        vector_tool, summary_tool = get_doc_tools(str(paper_path), paper_name)
        registry[vector_tool.metadata.name] = vector_tool
        registry[summary_tool.metadata.name] = summary_tool
    return registry


tool_registry = build_tool_registry()
tool_names = list(tool_registry.keys())
print("Loaded tools:", tool_names)


def extract_json_object(text: str) -> dict[str, Any]:
    # qwen-max 有时会返回 ```json 代码块，有时直接返回 JSON。
    # 这里统一把最外层 JSON 对象提出来，再交给 Pydantic 做结构校验。
    cleaned = text.strip()
    cleaned = cleaned.replace("```json", "```").replace("```JSON", "```")
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`").strip()
    match = re.search(r"\{.*\}", cleaned, re.S)
    if not match:
        raise ValueError(f"Could not find JSON object in model output:\n{cleaned}")
    return json.loads(match.group(0))


def complete_json(prompt: str) -> dict[str, Any]:
    response = llm.complete(prompt)
    return extract_json_object(response.text)


def call_tool(tool_name: str, query: str, page_numbers: Optional[list[str]] = None) -> dict[str, Any]:
    # 课程里的工具分成两类：
    # 1. vector_tool_* 需要 query，可选 page_numbers
    # 2. summary_tool_* 通常只需要一个输入问题
    tool = tool_registry[tool_name]
    page_numbers = page_numbers or []

    attempts: list[dict[str, Any]]
    if tool_name.startswith("vector_tool_"):
        attempts = [
            {"query": query, "page_numbers": page_numbers},
            {"query": query},
            {"input": query},
        ]
    else:
        attempts = [
            {"input": query},
            {"query": query},
        ]

    last_error = None
    for kwargs in attempts:
        try:
            tool_output = tool.call(**kwargs)
            raw_output = getattr(tool_output, "raw_output", None)
            source_nodes = getattr(raw_output, "source_nodes", []) if raw_output is not None else []
            source_pages = [
                node.metadata.get("page_label")
                for node in source_nodes
                if hasattr(node, "metadata")
            ]
            return {
                "tool_name": tool_name,
                "query": query,
                "content": str(getattr(tool_output, "content", tool_output)),
                "source_pages": [p for p in source_pages if p],
            }
        except Exception as exc:  # noqa: BLE001
            last_error = exc

    raise RuntimeError(f"Tool call failed for {tool_name}: {last_error}")


def build_tool_catalog() -> str:
    # 这份目录会喂给 planner / critic / evaluator。
    # 它相当于让模型先知道“我手上有哪些工具，每个工具大概适合干什么”。
    lines = []
    for tool_name, tool in tool_registry.items():
        lines.append(f"- {tool_name}: {tool.metadata.description}")
    return "\n".join(lines)


tool_catalog = build_tool_catalog()


## 1. Define a small evaluation runner


In [ ]:
class EvalPlanStep(BaseModel):
    tool_name: str
    query: str
    rationale: str


class EvalPlan(BaseModel):
    steps: list[EvalPlanStep]
    reasoning: str


class AgenticEvalResult(BaseModel):
    answer_correctness: int = Field(description="1-5 score for answer correctness.")
    groundedness: int = Field(description="1-5 score for how well the answer is supported by retrieved evidence.")
    planning_quality: int = Field(description="1-5 score for whether the plan/tool choices made sense.")
    tool_efficiency: int = Field(description="1-5 score for whether the workflow used a reasonable number of tools.")
    overall_score: int = Field(description="1-5 overall score.")
    strengths: list[str]
    weaknesses: list[str]
    improvement_suggestions: list[str]


In [ ]:
def build_eval_plan(question: str) -> EvalPlan:
    prompt = f'''
    You are planning a compact retrieval workflow over research-paper tools.

    Available tools:
    {tool_catalog}

    User question:
    {question}

    Return JSON only:
    {{
      "reasoning": "...",
      "steps": [
        {{
          "tool_name": "...",
          "query": "...",
          "rationale": "..."
        }}
      ]
    }}
    '''
    plan_dict = complete_json(prompt)
    return EvalPlan.model_validate(plan_dict)


def run_eval_case(question: str) -> dict[str, Any]:
    plan = build_eval_plan(question)
    trace = []

    for index, step in enumerate(plan.steps, start=1):
        result = call_tool(step.tool_name, step.query)
        trace.append(
            {
                "round": index,
                "tool_name": step.tool_name,
                "query": step.query,
                "rationale": step.rationale,
                "content": result["content"],
                "source_pages": result["source_pages"],
            }
        )

    evidence_text = "\n\n".join(
        f"Round {item['round']} | Tool={item['tool_name']} | Query={item['query']}\n{item['content']}"
        for item in trace
    )

    answer_prompt = f'''
    Answer the following question using only the retrieved evidence.

    Question:
    {question}

    Retrieved evidence:
    {evidence_text}
    '''
    answer = llm.complete(answer_prompt).text

    return {
        "plan": plan,
        "trace": trace,
        "answer": answer,
    }


## 2. Define an evaluation rubric


In [ ]:
def evaluate_agentic_run(question: str, reference_points: list[str], run_result: dict[str, Any]) -> AgenticEvalResult:
    trace_summary = "\n".join(
        f"- Tool={item['tool_name']} | Query={item['query']} | Pages={item['source_pages']} | Why={item['rationale']}"
        for item in run_result["trace"]
    )
    prompt = f'''
    You are evaluating an agentic RAG workflow.

    Question:
    {question}

    Reference points that a strong answer should cover:
    {reference_points}

    Planner reasoning:
    {run_result["plan"].reasoning}

    Retrieval trace:
    {trace_summary}

    Final answer:
    {run_result["answer"]}

    Score the system on:
    - answer_correctness
    - groundedness
    - planning_quality
    - tool_efficiency
    - overall_score

    Return JSON only:
    {{
      "answer_correctness": 5,
      "groundedness": 5,
      "planning_quality": 5,
      "tool_efficiency": 5,
      "overall_score": 5,
      "strengths": ["..."],
      "weaknesses": ["..."],
      "improvement_suggestions": ["..."]
    }}
    '''
    eval_dict = complete_json(prompt)
    return AgenticEvalResult.model_validate(eval_dict)


In [ ]:
# 这些问题都沿用了 Lesson 3 / 4 的论文主题，只是换成“可评测”的形式。
eval_examples = [
    {
        "question": "Give me a summary of both Self-RAG and LongLoRA, and explain their main difference.",
        "reference_points": [
            "Self-RAG is about retrieval-aware generation and self-reflection.",
            "LongLoRA is about efficient long-context adaptation/training.",
            "They solve different core problems rather than being two variants of the same retrieval method.",
        ],
    },
    {
        "question": "Tell me about the evaluation dataset used in MetaGPT and compare it against SWE-Bench style evaluation goals if mentioned elsewhere.",
        "reference_points": [
            "MetaGPT evaluation should be grounded in the paper evidence.",
            "The answer should distinguish software-agent evaluation from software-benchmark style evaluation.",
            "If evidence is missing, the answer should say so rather than hallucinate.",
        ],
    },
]

eval_examples


In [ ]:
eval_runs = []
for example in eval_examples:
    run_result = run_eval_case(example["question"])
    evaluation = evaluate_agentic_run(
        example["question"],
        example["reference_points"],
        run_result,
    )
    eval_runs.append(
        {
            "question": example["question"],
            "run_result": run_result,
            "evaluation": evaluation,
        }
    )

len(eval_runs)


In [ ]:
import pandas as pd

# 把结构化评测结果整理成 DataFrame，方便横向比较多个问题。
eval_rows = []
for item in eval_runs:
    evaluation = item["evaluation"]
    eval_rows.append(
        {
            "question": item["question"],
            "answer_correctness": evaluation.answer_correctness,
            "groundedness": evaluation.groundedness,
            "planning_quality": evaluation.planning_quality,
            "tool_efficiency": evaluation.tool_efficiency,
            "overall_score": evaluation.overall_score,
        }
    )

pd.DataFrame(eval_rows)


In [ ]:
# 详细查看第一条样本的计划、检索轨迹和评测意见。
sample = eval_runs[0]
print("QUESTION:")
print(sample["question"])
print("\nPLAN:")
print(sample["run_result"]["plan"].model_dump_json(indent=2))
print("\nANSWER:")
print(sample["run_result"]["answer"])
print("\nEVALUATION:")
print(sample["evaluation"].model_dump_json(indent=2))
